# Tutorials

## 🚀 Quick start - install

To install `covmats`, the easiest way is through `pip`:

```bash
    pip install covmats
```
Or alternatively using `conda`

```bash
    conda install covmats
```

You might also clone the repository and install from source

```bash
    pip install -e .
```

Once the installation is done, we can start using covariance matrices. There are various representations:
- CovarianceMatrix
- CovViaDiagonal
- CovViaCholesky
- CovViaEigendecomposition
- CovViaEnsemble
- CovViaFFT
- CovViaPrecision
- CovViaSparseCholesky
- CovViaSparsePrecision


Convention, Q for the precision and Sigma for 

Let's start by importing `numpy`, `scipy` and `covmats` for the tests

In [1]:
import numpy as np
import scipy as sp
import covmats

## Diagonal matrix

Let's start with the simple case of a diagonal matrix. 

In [2]:
d = [1, 2, 3]
A33 = np.diag(d)  # a diagonal covariance matrix
x = [4, -2, 5]  # a point of interest
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=A33)
dist.pdf(x)

np.float64(4.9595685102808205e-08)

It is compatible with the stats API from scipy since the base class inherit from `Covariance`.

In [3]:
cov_diag33 = covmats.CovViaDiagonal(d)
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=cov_diag33)
dist.pdf(x)

np.float64(4.9595685102808205e-08)

In [4]:
cov_diag33.rank, cov_diag33.log_pdet, cov_diag33.get_trace()

(np.int64(3), np.float64(1.791759469228055), 6.0)

- It also behaves as a Linearoperator, supporting matrix-vector, matrix-matrix and solve operations

In [5]:
v3 = np.array([1.0, 2.0, 3.0])
cov_diag33 @ v3

array([1., 4., 9.])

In [6]:
assert np.allclose(np.linalg.inv(A33) @ v3, np.ones_like(v3))

In [7]:
assert np.allclose(cov_diag33.solve(v3), np.ones_like(v3))

In [8]:
# product with a matrix (3, 2)
V32 = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]]).T
cov_diag33 @ V32

array([[1., 1.],
       [4., 4.],
       [9., 9.]])

In [9]:
cov_diag33.solve(V32)

array([[1., 1.],
       [1., 1.],
       [1., 1.]])

In [10]:
cov_diag33.whiten(v3)

array([1.        , 1.41421356, 1.73205081])

In [11]:
V32.T * cov_diag33._diagonal

array([[1., 4., 9.],
       [1., 4., 9.]])

In [12]:
cov_diag33.whiten(V32.T).shape

(2, 3)

## Cholesky

In [13]:
rng = np.random.default_rng(2026)
n = 5
A55 = rng.random(size=(n, n))
A55 = A55 @ A55.T  # make the covariance symmetric positive definite
x = rng.random(size=n)

v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])

In [14]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
L = np.linalg.cholesky(A55)
cov_cho55 = covmats.CovViaCholesky(L)
np.allclose(cov_cho55.log_pdet, np.linalg.slogdet(A55)[-1])
cov_cho55.rank, cov_cho55.log_pdet, cov_cho55.get_trace()

(np.int64(5), np.float64(-10.38508697140242), 8.160290791028103)

In [15]:
V52 = np.tile(v5.reshape(-1, 1), 2)

In [16]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_cho55)
dist.pdf(x)

np.float64(0.000619698809563932)

In [17]:
x = np.random.default_rng(676878).normal(size=5)
res = cov_cho55.whiten(x)
ref = sp.linalg.solve_triangular(cov_cho55._cho_factor, x, lower=True)
np.allclose(res, ref)

True

In [18]:
np.testing.assert_allclose(cov_cho55.colorize(cov_cho55.whiten(x)), x)

In [19]:
res = cov_cho55.colorize(x)
res

array([0.41404402, 0.4899987 , 0.32315677, 0.33313679, 0.84757021])

In [20]:
cov_cho55.colorize(np.random.default_rng(676878).normal(size=5))

array([0.41404402, 0.4899987 , 0.32315677, 0.33313679, 0.84757021])

In [21]:
cov_cho55.shape == (5, 5)

True

In [22]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_cho55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [23]:
cov_cho55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [24]:
cov_cho55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [25]:
cov_cho55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [26]:
cov_cho55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

## Working with precision (covariance inverse)

It is also possible to work with the inverse of the covariance matrix, namely the precision matrix

In [27]:
sp.linalg.cholesky(sp.linalg.inv(A55), lower=True)

array([[13.15098867,  0.        ,  0.        ,  0.        ,  0.        ],
       [-5.14721689,  4.06286381,  0.        ,  0.        ,  0.        ],
       [-7.98311558, -3.80212151,  1.92255136,  0.        ,  0.        ],
       [16.33812215, -1.04474489, -2.55077291,  1.83920109,  0.        ],
       [-7.92394486,  2.38453849, -0.51788415, -1.59648599,  0.95234685]])

In [28]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
cov_Q55 = covmats.CovViaPrecisionCholesky(
    sp.linalg.cholesky(sp.linalg.inv(A55), lower=True)
)
# Sanity checks
np.allclose(cov_Q55.log_pdet, np.linalg.slogdet(A55)[-1])
np.testing.assert_allclose(cov_Q55.todense(), A55)
cov_Q55.rank, cov_Q55.log_pdet, cov_Q55.get_trace()

(np.int64(5), np.float64(-10.385086971402444), 8.160290791027672)

In [29]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_Q55)
dist.pdf(x)

np.float64(3.3277799725037206e-241)

In [30]:
rng = np.random.default_rng(2026)
n = 3
A = rng.random(size=(n, n))
cov_array = A @ A.T  # make matrix symmetric positive definite
precision = np.linalg.inv(cov_array)
cov_object = covmats.CovViaPrecisionCholesky(sp.linalg.cholesky(precision, lower=True))
x = rng.multivariate_normal(np.zeros(n), cov_array, size=(10000))
x_ = cov_object.whiten(x)
# near-identity covariance is expected
np.testing.assert_allclose(np.cov(x_, rowvar=False), np.eye(3), atol=0.01)

In [31]:
x = np.random.default_rng(676878).normal(size=5)

# This is not True => need to check if this is correct ???
np.allclose(cov_Q55.whiten(x), cov_cho55.whiten(x))

False

In [32]:
np.testing.assert_allclose(cov_Q55.colorize(cov_Q55.whiten(x)), x)

In [33]:
cov_Q55.whiten(x)

array([31.80930111,  5.58426935, -7.8710699 ,  0.81242828,  1.73424998])

In [34]:
cov_Q55.shape == (5, 5)

True

In [35]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_Q55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [36]:
cov_Q55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [37]:
cov_Q55.prec = 8

In [38]:
sp.sparse.csc_array([[1.0, 1.0]]).todense()

array([[1., 1.]])

In [39]:
cov_Q55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [40]:
cov_Q55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [41]:
cov_Q55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

## Sparse cholesky

In [42]:
import numpy as np
from scipy.sparse import lil_matrix, csr_matrix

# Matrix size
n = 10

# Initialize a sparse matrix
cov_matrix = lil_matrix((n, n))

# Fill the diagonal with variances (non-zero values)
np.random.seed(42)  # for reproducibility
variances = np.random.uniform(0.5, 2.0, n)
for i in range(n):
    cov_matrix[i, i] = variances[i]

# Add some random non-zero covariances
for i in range(n):
    for j in range(i + 1, n):
        if np.random.rand() < 0.1:  # 10% chance of non-zero covariance
            cov = np.random.uniform(-0.5, 0.5)
            cov_matrix[i, j] = cov
            cov_matrix[j, i] = cov  # symmetry

# Convert to CSR format for efficient use
cov_matrix = cov_matrix.tocsr()

print("Sparse covariance matrix (CSR format):")
print(cov_matrix.toarray())  # Dense display for visualization

Sparse covariance matrix (CSR format):
[[ 1.06181018  0.46990985  0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.46990985  1.92607146  0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.          0.          1.59799091  0.          0.10754485  0.
   0.44888554  0.          0.          0.        ]
 [ 0.          0.          0.          1.39798773  0.18423303  0.
   0.          0.          0.4093204   0.        ]
 [ 0.          0.          0.10754485  0.18423303  0.73402796  0.
   0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.73399178
   0.          0.          0.          0.        ]
 [ 0.          0.          0.44888554  0.          0.          0.
   0.58712542  0.          0.         -0.30401714]
 [ 0.          0.          0.          0.          0.          0.
   0.          1.79926422 -0.17466967  0.        ]
 [ 0.          0.

In [51]:
cov_cho_dense = covmats.CovViaCholesky(
    sp.linalg.cholesky(cov_matrix.todense(), lower=True)
)

In [44]:
from covmats._sparse_helpers import sparse_cholesky

In [45]:
F = sparse_cholesky(cov_matrix)

In [63]:
cov_cho_sparse.todense()

matrix([[ 1.06181018,  0.46990985,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.46990985,  1.92607146,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.73399178,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  1.56210887, -0.30401714,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , -0.30401714,  0.58712542,
          0.44888554,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.44888554,
          1.59799091,  0.10754485,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.10754485,  0.7340279

In [64]:
cov_cho_dense.todense()

array([[ 1.06181018e+00,  4.69909852e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 4.69909852e-01,  1.92607146e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.59799091e+00,
         0.00000000e+00,  1.07544852e-01,  0.00000000e+00,
         4.48885537e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.39798773e+00,  1.84233027e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  4.09320402e-01,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  1.07544852e-01,
         1.84233027e-01,  7.34027961e-01,  0.00000000e+00,
        -1.32513196e-18,  0.00000000e+00, -4.58200729e-18,
         0.

In [62]:
cov_cho_sparse = covmats.CovViaSparseCholesky(sparse_cholesky(cov_matrix))
np.testing.assert_allclose(cov_cho_sparse.todense(), cov_cho_dense.todense())

IndexError: boolean index did not match indexed array along axis 0; size of axis is 100 but size of corresponding boolean axis is 1

In [57]:
cov_cho_sparse.todense().shape, cov_cho_dense.todense().shape

((10, 10), (10, 10))

In [ ]:
np.testing.assert_allclose(
    cov_cho_sparse.solve(np.eye(10)), sparse_cholesky(cov_matrix).inv().todense()
)
# todo, compare with dense version

## Eigen 

Eigen is great ! It can be generated from a dense matrix

In [47]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
cov_eig55 = covmats.eigen_factorize_cov_mat(cov_Q55, n_pc=4, random_state=376)
# Sanity checks
np.allclose(cov_eig55.log_pdet, np.linalg.slogdet(A55)[-1])
np.testing.assert_allclose(cov_eig55.todense(), A55, rtol=0.001)
cov_eig55.rank, cov_eig55.log_pdet, cov_eig55.get_trace()

(np.int64(4), np.float64(-3.984708722959997), 8.1586298621171)

In [48]:
dist = sp.stats.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_eig55)
dist.pdf(x)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1 is different from 5)

In [ ]:
x = np.random.default_rng(676878).normal(size=5)

# This is not True => need to check if this is correct ???
# np.allclose(cov_eig55.whiten(x), cov_Q55.whiten(x))

False

In [ ]:
cov_eig55.whiten(x)

array([-0.80707474,  3.55211942, -5.8464596 ,  1.7129155 ,  5.63621083])

In [ ]:
(cov_eig55._v.T / np.sqrt(cov_eig55._w)) @ x

array([-0.32871033,  2.49980091, -2.60598443, -8.30678963])

In [ ]:
# (cov_eig55._v.T * (1.0 / np.sqrt(cov_eig55._w + eps))).shape

NameError: name 'eps' is not defined

In [ ]:
x.shape

(5,)

In [ ]:
x

array([ 0.4309492 , -0.03543374, -0.92023751,  2.0224398 ,  1.82102769])

In [ ]:
# Whitening operator (n x n, implicit)
eps = 1e-6
x2 = ((cov_eig55._v.T * (1.0 / np.sqrt(cov_eig55._w + eps))).T @ cov_eig55._v.T) @ z

NameError: name 'z' is not defined

In [ ]:
x2

array([-0.26345927,  0.24639157, -0.49901551,  1.1380487 ,  2.24964905])

In [ ]:
z = ((cov_eig55._v.T * np.sqrt(cov_eig55._w)).T @ cov_eig55._v.T) @ x

In [ ]:
z

array([-0.80708197,  3.55202411, -5.84634555,  1.7129135 ,  5.63614557])

In [ ]:
(cov_eig55._v.T * np.sqrt(cov_eig55._w))

array([[-8.77371185e-01, -1.35666962e+00, -1.73293123e+00,
        -1.01442716e+00, -9.19486533e-01],
       [ 1.28799005e-01, -3.74734375e-01, -3.74400571e-02,
         2.75321861e-03,  4.97532369e-01],
       [ 3.68961646e-01,  2.27862189e-02,  1.72362264e-02,
        -3.10745016e-01, -7.53362826e-02],
       [-1.05547765e-02, -9.58970277e-02,  1.11975417e-01,
         1.45443402e-03, -6.10776470e-02]])

In [ ]:
np.testing.assert_allclose(cov_Q55.colorize(cov_Q55.whiten(x)), x)

In [ ]:
cov_Q55.whiten(x)

array([31.80930111,  5.58426935, -7.8710699 ,  0.81242828,  1.73424998])

In [ ]:
cov_Q55.shape == (5, 5)

True

In [ ]:
v5 = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_Q55 @ v5

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [ ]:
cov_Q55.solve(v5)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [ ]:
cov_Q55 @ V52

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [ ]:
cov_Q55.solve(V52)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [ ]:
cov_Q55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

In [ ]:
covmats.eigen_factorize_cov_mat(cov_Q55)

eigen_factorize_cov_mat,
generate_dense_matrix,
get_explained_var,
get_matrix_eigen_factorization,

## SVD

In [ ]:
sp.sparse.linalg.svds()

## Kernel based Covariances

Lib such as gstools, gstlearn, etc. provide kernels

### FFT

### Hirearchical

## Ensemble of realizations

## Sparse precision and cholesky

- Talk about SPDE, large scale applications

# Drawing samples from the multivariate normal $\mathcal{N}\left(\mathbf{0}, \mathbf{Q} \right)$

To draw samples

In [ ]:
covd = covmats.CovViaDiagonal(np.array([5.0, 10.0, 15.0]))
rng_seed = 42
covd.sample_mvnormal(shape=[2], random_state=rng_seed)

array([[ 1.11068661, -0.43723011,  2.50848692],
       [ 3.40559829, -0.74045799, -0.90680853]])

In [ ]:
x = covd.sample_mvnormal(shape=[2, 4], random_state=rng_seed)
x

array([[[ 1.11068661, -0.43723011,  2.50848692],
        [ 3.40559829, -0.74045799, -0.90680853],
        [ 3.53122721,  2.4268417 , -1.81826648],
        [ 1.21320114, -1.46545542, -1.80376358]],

       [[ 0.54104409, -6.05032338, -6.68057804],
        [-1.25731314, -3.20285323,  1.21707469],
        [-2.03040356, -4.46609644,  5.67643327],
        [-0.50485116,  0.21354293, -5.518026  ]]])

In [ ]:
x.shape

(2, 4, 3)

In [ ]:
cov_cho = covmats.CovViaCholesky(sp.linalg.cholesky(covd.todense()))
cov_cho.sample_mvnormal(shape=[2], random_state=rng_seed)

array([[ 1.11068661, -0.43723011,  2.50848692],
       [ 3.40559829, -0.74045799, -0.90680853]])

In [ ]:
cov_cho.sample_mvnormal(shape=[2, 4], random_state=rng_seed)

array([[[ 1.11068661, -0.43723011,  2.50848692],
        [ 3.40559829, -0.74045799, -0.90680853],
        [ 3.53122721,  2.4268417 , -1.81826648],
        [ 1.21320114, -1.46545542, -1.80376358]],

       [[ 0.54104409, -6.05032338, -6.68057804],
        [-1.25731314, -3.20285323,  1.21707469],
        [-2.03040356, -4.46609644,  5.67643327],
        [-0.50485116,  0.21354293, -5.518026  ]]])